In [48]:
import pandas as pd
import numpy as np
import kagglehub
import os
from datetime import datetime

In [49]:
log = {}

In [50]:
def collect_from_kaggle(kaggle_path: str) -> pd.DataFrame:
    path = kagglehub.dataset_download(kaggle_path)
    file_dir = os.listdir(path)
    df_path = f"{path}/{file_dir[0]}"
    df = pd.read_csv(df_path)

    return df

In [51]:
df = collect_from_kaggle("shashanknecrothapa/ames-housing-dataset")

In [52]:
def check_outliers_overall(df: pd.DataFrame, col: str) -> dict:
    lq = np.percentile(df[col], 25)
    uq = np.percentile(df[col], 75)
    iqr = uq-lq
    lt = lq - (1.5*iqr)
    ut = uq + (1.5*iqr)

    lower_mask = df[col] < lt
    upper_mask = df[col] > ut
    overall_mask = (df[col] < lt) | (df[col] > ut)

    perc_under = lower_mask.sum()/len(df)
    perc_over = upper_mask.sum()/len(df)
    perc_all = overall_mask.sum()/len(df)

    perc_dict = {'percentage_under_threshold': perc_under,
                 'percentage_over_threshold': perc_over,
                 'percentage_anomalous': perc_all}

    return perc_dict

In [53]:
print(check_outliers_overall(df, 'SalePrice'))

{'percentage_under_threshold': np.float64(0.0), 'percentage_over_threshold': np.float64(0.04675767918088737), 'percentage_anomalous': np.float64(0.04675767918088737)}


In [54]:
def drop_outliers(df: pd.DataFrame, col: str, log_dict: dict) -> tuple:
    lq = np.percentile(df[col], 25)
    uq = np.percentile(df[col], 75)
    iqr = uq-lq
    lt = lq - (1.5*iqr)
    ut = uq + (1.5*iqr)

    overall_mask = (df[col] >= lt) & (df[col] <= ut)
    log_dict[datetime.now()] = {'action': 'drop_outliers', 'col': col, 'rows_affected': len(df) - overall_mask.sum()}

    return df.loc[overall_mask], log_dict

df2, log = drop_outliers(df, 'SalePrice', log)

print(check_outliers_overall(df2, 'SalePrice'))

{'percentage_under_threshold': np.float64(0.0007160759040458289), 'percentage_over_threshold': np.float64(0.021124239169351952), 'percentage_anomalous': np.float64(0.02184031507339778)}


In [55]:
def cap_outliers(df: pd.DataFrame, col: str, log_dict: dict) -> tuple:
    lq = np.percentile(df[col], 25)
    uq = np.percentile(df[col], 75)
    iqr = uq-lq
    lt = lq - (1.5*iqr)
    ut = uq + (1.5*iqr)
    overall_mask = (df[col] >= lt) & (df[col] <= ut)
    log_dict[datetime.now()] = {'action': 'cap_outliers', 'col': col, 'rows_affected': len(df) - overall_mask.sum()}

    df[col] = df[col].clip(lower = lt, upper = ut)

    return df, log_dict

In [56]:
df2, log = cap_outliers(df, 'SalePrice', log)

print(check_outliers_overall(df2, 'SalePrice'))

{'percentage_under_threshold': np.float64(0.0), 'percentage_over_threshold': np.float64(0.0), 'percentage_anomalous': np.float64(0.0)}


In [57]:
def log_transform_col(df: pd.DataFrame, col: str, log_dict: dict) -> tuple:
    df[col] = np.log1p(df[col])
    log_dict[datetime.now()] = {'action': 'cap_outliers', 'col': col, 'rows_affected': len(df)}
    
    return df, log_dict

In [58]:
df2, log = log_transform_col(df, 'SalePrice', log)
print(check_outliers_overall(df2, 'SalePrice'))

{'percentage_under_threshold': np.float64(0.00955631399317406), 'percentage_over_threshold': np.float64(0.0), 'percentage_anomalous': np.float64(0.00955631399317406)}


In [59]:
print(log)

{datetime.datetime(2026, 9, 13, 0, 7, 21, 332633): {'action': 'drop_outliers', 'col': 'SalePrice', 'rows_affected': np.int64(137)}, datetime.datetime(2026, 9, 13, 0, 7, 21, 365101): {'action': 'cap_outliers', 'col': 'SalePrice', 'rows_affected': np.int64(137)}, datetime.datetime(2026, 9, 13, 0, 7, 21, 390241): {'action': 'cap_outliers', 'col': 'SalePrice', 'rows_affected': 2930}}
